# SVMs — Support Vector Machines
**Mathematics for Machine Learning, Ch 12**

This notebook implements SVMs from scratch and demonstrates the kernel trick.
Topics covered:
1. Hard-margin SVM: linearly separable case, solved via QP
2. Soft-margin SVM: non-separable case, effect of C parameter
3. Kernel trick: XOR data, RBF kernel in input vs feature space
4. Kernel comparison: linear, polynomial (degree 3), RBF

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from scipy.optimize import minimize
from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
plt.rcParams.update({'figure.dpi': 100, 'font.size': 11})

---
## 1. Hard-Margin SVM

For linearly separable data, the hard-margin SVM solves:
$$\min_{w, b} \frac{1}{2}\|w\|^2 \quad \text{subject to} \quad y_n(w^\top x_n + b) \geq 1 \quad \forall n$$

This is equivalent to maximizing the **margin width** $= 2/\|w\|$.

**Dual formulation** (only inner products appear — the key to kernelization):
$$\max_\alpha \sum_n \alpha_n - \frac{1}{2}\sum_{n,m} \alpha_n \alpha_m y_n y_m x_n^\top x_m \quad \text{s.t.} \quad \alpha_n \geq 0,\ \sum_n \alpha_n y_n = 0$$

We solve the dual using `scipy.optimize.minimize` (treating it as minimization of the negated dual).

In [ ]:
# Generate linearly separable 2D data
N_pos = 30
N_neg = 30
X_pos = np.random.randn(N_pos, 2) + np.array([2.0, 2.0])
X_neg = np.random.randn(N_neg, 2) + np.array([-2.0, -2.0])
X_sep = np.vstack([X_pos, X_neg])
y_sep = np.array([1] * N_pos + [-1] * N_neg, dtype=float)
N_sep = len(X_sep)

print(f'Data: {N_pos} positive, {N_neg} negative, linearly separable')

In [ ]:
def solve_hard_margin_svm(X, y):
    """
    Solve the hard-margin SVM dual via scipy.optimize.minimize.
    Dual: max Σα_n - 0.5 Σ_{n,m} α_n α_m y_n y_m x_n·x_m
    s.t. α_n >= 0, Σ α_n y_n = 0
    Returns: alpha, w, b
    """
    N = len(y)
    # Gram matrix Q[n,m] = y_n y_m x_n·x_m
    Q = np.outer(y, y) * (X @ X.T)

    # Objective: minimize -dual = -Σα + 0.5 α^T Q α
    def objective(alpha):
        return 0.5 * alpha @ Q @ alpha - alpha.sum()

    def grad_objective(alpha):
        return Q @ alpha - np.ones(N)

    # Constraints and bounds
    constraints = {'type': 'eq', 'fun': lambda a: np.dot(a, y)}
    bounds = [(0, None)] * N
    alpha0 = np.zeros(N)

    result = minimize(objective, alpha0, jac=grad_objective,
                      method='SLSQP', bounds=bounds, constraints=constraints,
                      options={'ftol': 1e-9, 'maxiter': 1000})

    alpha = result.x
    alpha[alpha < 1e-5] = 0.0  # threshold near-zeros

    # Recover primal: w = Σ α_n y_n x_n
    w = (alpha * y) @ X

    # Recover bias from support vectors (averaged for stability)
    sv_mask = alpha > 1e-5
    b_vals  = y[sv_mask] - X[sv_mask] @ w
    b       = b_vals.mean()

    return alpha, w, b, sv_mask


alpha_hm, w_hm, b_hm, sv_mask_hm = solve_hard_margin_svm(X_sep, y_sep)

margin_width = 2.0 / np.linalg.norm(w_hm)
n_support_vectors = sv_mask_hm.sum()

print(f'w = {w_hm.round(4)}')
print(f'b = {b_hm:.4f}')
print(f'Margin width = 2/||w|| = {margin_width:.4f}')
print(f'Number of support vectors: {n_support_vectors}')

In [ ]:
def plot_svm_2d(ax, X, y, w, b, sv_mask, title, C=None):
    """Plot data, decision boundary, margins, and support vectors."""
    # Decision boundary and margins
    x1_min, x1_max = X[:, 0].min() - 1, X[:, 0].max() + 1
    x2_min, x2_max = X[:, 1].min() - 1, X[:, 1].max() + 1
    xx = np.linspace(x1_min, x1_max, 300)

    # Decision boundary: w[0]*x1 + w[1]*x2 + b = 0  →  x2 = -(w[0]*x1 + b) / w[1]
    if abs(w[1]) > 1e-8:
        x2_db  = -(w[0] * xx + b) / w[1]
        x2_mp  = -(w[0] * xx + b - 1) / w[1]  # +1 margin
        x2_mm  = -(w[0] * xx + b + 1) / w[1]  # -1 margin
    else:
        x2_db = x2_mp = x2_mm = np.zeros_like(xx)

    ax.fill_between(xx, x2_mp, x2_mm, alpha=0.1, color='gray', label='margin')
    ax.plot(xx, x2_db, 'k-',  lw=2,   label='decision boundary')
    ax.plot(xx, x2_mp, 'k--', lw=1.2, label='+1 margin')
    ax.plot(xx, x2_mm, 'k--', lw=1.2, label='-1 margin')

    # Data points
    pos = y == 1
    neg = y == -1
    ax.scatter(X[pos, 0], X[pos, 1], c='steelblue', s=30, alpha=0.7, label='+1')
    ax.scatter(X[neg, 0], X[neg, 1], c='tomato',    s=30, alpha=0.7, label='-1')

    # Support vectors
    ax.scatter(X[sv_mask, 0], X[sv_mask, 1], s=180, facecolors='none',
               edgecolors='black', lw=2, zorder=5, label='support vectors')

    ax.set_xlim(x1_min, x1_max)
    ax.set_ylim(x2_min, x2_max)
    ax.set_title(title)
    ax.legend(fontsize=8, loc='upper left')
    ax.set_aspect('equal')


fig, ax = plt.subplots(figsize=(6, 6))
plot_svm_2d(ax, X_sep, y_sep, w_hm, b_hm, sv_mask_hm,
            f'Hard-Margin SVM\nMargin = {margin_width:.2f}, SVs = {n_support_vectors}')
plt.tight_layout()
plt.show()

---
## 2. Soft-Margin SVM

When data is not linearly separable, introduce **slack variables** $\xi_n \geq 0$:
$$\min_{w, b, \xi} \frac{1}{2}\|w\|^2 + C\sum_n \xi_n \quad \text{s.t.} \quad y_n(w^\top x_n + b) \geq 1 - \xi_n,\ \xi_n \geq 0$$

- **Large C**: heavy penalty on violations → narrow margin, risk of overfitting outliers
- **Small C**: slack allowed → wider margin, more misclassifications tolerated

Equivalently, this is **L2-regularized ERM with hinge loss**: $\max(0, 1 - y_n f(x_n))$.

In [ ]:
# Make data non-separable by adding overlapping noise
np.random.seed(10)
X_noisy = X_sep.copy()
y_noisy = y_sep.copy()

# Add 10 mislabeled / noisy points
n_noise = 10
noise_idx = np.random.choice(len(X_noisy), n_noise, replace=False)
X_noisy[noise_idx] += np.random.randn(n_noise, 2) * 2.5  # push some to wrong side
y_noisy[noise_idx] *= -1  # flip labels of some

# Fit with different C values using sklearn
C_values = [0.01, 0.1, 1.0, 10.0, 100.0]

fig, axes = plt.subplots(1, len(C_values), figsize=(18, 4))

x1_grid = np.linspace(X_noisy[:, 0].min() - 1, X_noisy[:, 0].max() + 1, 200)
x2_grid = np.linspace(X_noisy[:, 1].min() - 1, X_noisy[:, 1].max() + 1, 200)
xx, yy  = np.meshgrid(x1_grid, x2_grid)
grid    = np.c_[xx.ravel(), yy.ravel()]

for ax, C in zip(axes, C_values):
    svm = SVC(kernel='linear', C=C)
    svm.fit(X_noisy, y_noisy)

    # Decision regions
    Z = svm.predict(grid).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.15, cmap=ListedColormap(['tomato', 'steelblue']))
    ax.contour(xx, yy, svm.decision_function(grid).reshape(xx.shape),
               levels=[-1, 0, 1], linestyles=['--', '-', '--'],
               colors='black', linewidths=[1, 2, 1])

    # Data
    pos, neg = y_noisy == 1, y_noisy == -1
    ax.scatter(X_noisy[pos, 0], X_noisy[pos, 1], c='steelblue', s=20, alpha=0.8)
    ax.scatter(X_noisy[neg, 0], X_noisy[neg, 1], c='tomato',    s=20, alpha=0.8)

    # Support vectors
    sv = svm.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=100, facecolors='none',
               edgecolors='black', lw=1.5, zorder=5)

    w     = svm.coef_[0]
    margin = 2.0 / np.linalg.norm(w)
    ax.set_title(f'C={C}\nmargin={margin:.2f}, SVs={len(sv)}')
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('Soft-Margin SVM: Effect of Regularization Parameter C', fontweight='bold')
plt.tight_layout()
plt.show()

print('C interpretation:')
print('  Small C → wide margin, more violations tolerated (high bias, low variance)')
print('  Large C → narrow margin, fewer violations (low bias, high variance / overfitting)')

---
## 3. Kernel Trick: XOR Data

XOR data is **not linearly separable** in $\mathbb{R}^2$, but becomes separable in a higher-dimensional feature space.

The **kernel trick** replaces the inner product $x_n^\top x_m$ with a kernel function:
$$k(x_n, x_m) = \phi(x_n)^\top \phi(x_m)$$
without ever computing $\phi$ explicitly.

The **RBF (Gaussian) kernel** corresponds to an infinite-dimensional feature space:
$$k(x, x') = \exp\!\left(-\frac{\|x - x'\|^2}{2\sigma^2}\right)$$

In [ ]:
# Generate XOR data
np.random.seed(5)
n_per_quad = 50
X_xor = np.vstack([
    np.random.randn(n_per_quad, 2) + [1.5,  1.5],   # quadrant +,+  → +1
    np.random.randn(n_per_quad, 2) + [-1.5, -1.5],  # quadrant -,-  → +1
    np.random.randn(n_per_quad, 2) + [1.5, -1.5],   # quadrant +,-  → -1
    np.random.randn(n_per_quad, 2) + [-1.5,  1.5],  # quadrant -,+  → -1
])
y_xor = np.array([1] * (2 * n_per_quad) + [-1] * (2 * n_per_quad), dtype=float)

# Fit: linear SVM (should fail) and RBF SVM (should succeed)
svm_linear_xor = SVC(kernel='linear', C=1.0)
svm_rbf_xor    = SVC(kernel='rbf', C=10.0, gamma=0.5)
svm_linear_xor.fit(X_xor, y_xor)
svm_rbf_xor.fit(X_xor, y_xor)

x1_range = np.linspace(X_xor[:, 0].min() - 0.5, X_xor[:, 0].max() + 0.5, 300)
x2_range = np.linspace(X_xor[:, 1].min() - 0.5, X_xor[:, 1].max() + 0.5, 300)
xx, yy   = np.meshgrid(x1_range, x2_range)
grid_xor = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
cmap_data = ListedColormap(['tomato', 'steelblue'])

for ax, svm, title in zip(
    axes,
    [svm_linear_xor, svm_rbf_xor],
    ['Linear SVM on XOR data\n(cannot separate — boundary is arbitrary)',
     'RBF Kernel SVM on XOR data\n(nonlinear boundary, perfect separation)']
):
    Z = svm.predict(grid_xor).reshape(xx.shape)
    ax.contourf(xx, yy, Z, alpha=0.2, cmap=cmap_data)
    df = svm.decision_function(grid_xor).reshape(xx.shape)
    ax.contour(xx, yy, df, levels=[-1, 0, 1],
               linestyles=['--', '-', '--'], colors='black', linewidths=[1, 2, 1])

    pos_mask = y_xor == 1
    ax.scatter(X_xor[pos_mask, 0],  X_xor[pos_mask, 1],  c='steelblue', s=25, alpha=0.8)
    ax.scatter(X_xor[~pos_mask, 0], X_xor[~pos_mask, 1], c='tomato',    s=25, alpha=0.8)

    sv = svm.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=100, facecolors='none',
               edgecolors='black', lw=1.5, zorder=5, label=f'SVs ({len(sv)})')

    acc = (svm.predict(X_xor) == y_xor).mean() * 100
    ax.set_title(f'{title}\nTraining acc: {acc:.0f}%')
    ax.legend(fontsize=9)

plt.suptitle('Kernel Trick: Linear vs RBF SVM on XOR Data', fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Visualize the RBF kernel boundary more carefully:
# Show that in the RKHS feature space, the boundary is approximately linear.
# We illustrate this by projecting to kernel PCA coordinates.

from sklearn.decomposition import KernelPCA

kpca = KernelPCA(n_components=2, kernel='rbf', gamma=0.5)
X_xor_kpca = kpca.fit_transform(X_xor)  # project to 2 kernel PCA components

# Fit linear SVM in the kernel PCA space
svm_lin_kpca = SVC(kernel='linear', C=10.0)
svm_lin_kpca.fit(X_xor_kpca, y_xor)

x1_kpca = np.linspace(X_xor_kpca[:, 0].min()-0.5, X_xor_kpca[:, 0].max()+0.5, 300)
x2_kpca = np.linspace(X_xor_kpca[:, 1].min()-0.5, X_xor_kpca[:, 1].max()+0.5, 300)
xx_k, yy_k = np.meshgrid(x1_kpca, x2_kpca)
Z_kpca = svm_lin_kpca.predict(np.c_[xx_k.ravel(), yy_k.ravel()]).reshape(xx_k.shape)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# Left: input space with curved RBF boundary
ax = axes[0]
Z_rbf = svm_rbf_xor.predict(grid_xor).reshape(xx.shape)
ax.contourf(xx, yy, Z_rbf, alpha=0.2, cmap=cmap_data)
ax.contour(xx, yy, svm_rbf_xor.decision_function(grid_xor).reshape(xx.shape),
           levels=[0], colors='black', linewidths=2)
pos_mask = y_xor == 1
ax.scatter(X_xor[pos_mask, 0],  X_xor[pos_mask, 1],  c='steelblue', s=30, alpha=0.8, label='+1')
ax.scatter(X_xor[~pos_mask, 0], X_xor[~pos_mask, 1], c='tomato',    s=30, alpha=0.8, label='-1')
ax.set_title('RBF SVM: curved boundary in input space $\\mathbb{R}^2$')
ax.legend()

# Right: kernel PCA space with linear boundary
ax = axes[1]
ax.contourf(xx_k, yy_k, Z_kpca, alpha=0.2, cmap=cmap_data)
ax.contour(xx_k, yy_k, svm_lin_kpca.decision_function(np.c_[xx_k.ravel(), yy_k.ravel()]).reshape(xx_k.shape),
           levels=[0], colors='black', linewidths=2)
ax.scatter(X_xor_kpca[pos_mask, 0],  X_xor_kpca[pos_mask, 1],  c='steelblue', s=30, alpha=0.8, label='+1')
ax.scatter(X_xor_kpca[~pos_mask, 0], X_xor_kpca[~pos_mask, 1], c='tomato',    s=30, alpha=0.8, label='-1')
ax.set_title('Linear boundary in kernel PCA feature space\n(approx. implicit feature space)')
ax.legend()

plt.suptitle('RBF Kernel: Curved Boundary in Input Space ↔ Linear in Feature Space', fontweight='bold')
plt.tight_layout()
plt.show()

---
## 4. Kernel Comparison: Linear, Polynomial (degree 3), RBF

Three common kernels for SVMs:

| Kernel | Formula | Feature space |
|--------|---------|---------------|
| Linear | $k(x,x') = x^\top x'$ | $\mathbb{R}^D$ (no mapping) |
| Polynomial (degree $p$) | $k(x,x') = (x^\top x' + c)^p$ | All monomials of degree $\leq p$ |
| RBF / Gaussian | $k(x,x') = \exp(-\|x-x'\|^2 / 2\sigma^2)$ | Infinite-dimensional |

A valid kernel must be **symmetric positive semidefinite** (Mercer's theorem).

In [ ]:
# Generate a more complex dataset for kernel comparison (two interleaved moons-like shapes)
from sklearn.datasets import make_moons

X_moon, y_moon = make_moons(n_samples=200, noise=0.2, random_state=42)
y_moon = 2 * y_moon - 1  # convert to {-1, +1}

# Scale features
scaler = StandardScaler()
X_moon_s = scaler.fit_transform(X_moon)

# Define kernels
kernels = [
    ('linear',      SVC(kernel='linear',      C=1.0),             'Linear\nC=1'),
    ('poly',        SVC(kernel='poly',         C=1.0, degree=3, coef0=1.0), 'Polynomial (d=3)\nC=1'),
    ('rbf C=0.1',   SVC(kernel='rbf',          C=0.1, gamma='scale'), 'RBF\nC=0.1 (smooth)'),
    ('rbf C=10',    SVC(kernel='rbf',          C=10.0, gamma='scale'), 'RBF\nC=10 (tight)'),
]

x1_range = np.linspace(X_moon_s[:, 0].min() - 0.5, X_moon_s[:, 0].max() + 0.5, 300)
x2_range = np.linspace(X_moon_s[:, 1].min() - 0.5, X_moon_s[:, 1].max() + 0.5, 300)
xx, yy   = np.meshgrid(x1_range, x2_range)
grid_moon = np.c_[xx.ravel(), yy.ravel()]

fig, axes = plt.subplots(1, 4, figsize=(18, 4.5))

for ax, (name, svm, title) in zip(axes, kernels):
    svm.fit(X_moon_s, y_moon)
    Z = svm.predict(grid_moon).reshape(xx.shape)

    ax.contourf(xx, yy, Z, alpha=0.2, cmap=ListedColormap(['tomato', 'steelblue']))
    ax.contour(xx, yy, svm.decision_function(grid_moon).reshape(xx.shape),
               levels=[-1, 0, 1], linestyles=['--', '-', '--'],
               colors='black', linewidths=[0.8, 2, 0.8])

    pos = y_moon == 1
    ax.scatter(X_moon_s[pos, 0],  X_moon_s[pos, 1],  c='steelblue', s=18, alpha=0.8)
    ax.scatter(X_moon_s[~pos, 0], X_moon_s[~pos, 1], c='tomato',    s=18, alpha=0.8)

    sv = svm.support_vectors_
    ax.scatter(sv[:, 0], sv[:, 1], s=80, facecolors='none',
               edgecolors='black', lw=1.2, zorder=5)

    acc  = (svm.predict(X_moon_s) == y_moon).mean() * 100
    n_sv = len(sv)
    ax.set_title(f'{title}\nacc={acc:.0f}%, SVs={n_sv}', fontsize=10)
    ax.set_xticks([]); ax.set_yticks([])

plt.suptitle('SVM Kernel Comparison: Decision Regions on Moons Dataset', fontweight='bold')
plt.tight_layout()
plt.show()

print('Interpretation:')
print('  Linear:         straight decision boundary — fails on non-linear data')
print('  Polynomial d=3: curved boundary — more flexible, can capture curvature')
print('  RBF small C:    smooth, wide boundary — higher bias, better generalization')
print('  RBF large C:    tight, jagged boundary — lower bias, risk of overfitting')

In [ ]:
# Cross-validation accuracy comparison across kernels and C values
from sklearn.model_selection import cross_val_score

kernel_configs = [
    ('linear, C=0.1',    SVC(kernel='linear', C=0.1)),
    ('linear, C=1',      SVC(kernel='linear', C=1.0)),
    ('poly d=3, C=1',    SVC(kernel='poly', degree=3, C=1.0, coef0=1.0)),
    ('poly d=3, C=10',   SVC(kernel='poly', degree=3, C=10.0, coef0=1.0)),
    ('RBF, C=0.1',       SVC(kernel='rbf', C=0.1, gamma='scale')),
    ('RBF, C=1',         SVC(kernel='rbf', C=1.0, gamma='scale')),
    ('RBF, C=10',        SVC(kernel='rbf', C=10.0, gamma='scale')),
]

print('5-fold cross-validation accuracy on moons dataset:')
print(f'{"Configuration":<22} {"Mean Acc":>10} {"Std":>8}')
print('-' * 44)
for name, clf in kernel_configs:
    scores = cross_val_score(clf, X_moon_s, y_moon, cv=5, scoring='accuracy')
    print(f'{name:<22} {scores.mean()*100:>9.1f}% {scores.std()*100:>7.1f}%')

---
## Summary

| Concept | Formula |
|---------|----------|
| Decision boundary | $w^\top x + b = 0$ |
| Prediction | $\hat{y} = \text{sign}(w^\top x^* + b)$ |
| Margin width | $2 / \|w\|$ |
| Hard-margin primal | $\min \frac{1}{2}\|w\|^2$ s.t. $y_n(w^\top x_n + b) \geq 1$ |
| Soft-margin primal | $\min \frac{1}{2}\|w\|^2 + C\sum \xi_n$ s.t. $y_n(w^\top x_n + b) \geq 1 - \xi_n$ |
| Hinge loss | $\max(0, 1 - y_n(w^\top x_n + b))$ |
| Dual (hard) | $\max \sum \alpha_n - \frac{1}{2}\sum_{n,m} \alpha_n \alpha_m y_n y_m x_n^\top x_m$ |
| Dual (soft) | same but $0 \leq \alpha_n \leq C$ |
| Kernel trick | $k(x_n, x_m) = \phi(x_n)^\top\phi(x_m)$ |
| RBF kernel | $k(x,x') = \exp(-\|x-x'\|^2 / 2\sigma^2)$ |
| Polynomial kernel | $k(x,x') = (x^\top x' + c)^p$ |
| KKT: $w$ recovery | $w^* = \sum_n \alpha_n y_n x_n$ |
| Sparsity | $\alpha_n = 0$ for non-support vectors |